# Task 2: Predict Future Stock Prices (Short-Term)
### DevelopersHub Corporation -- AI/ML Engineering Internship

---

## Problem Statement
The goal of this task is to use historical stock market data to predict the next day's closing price using supervised machine learning regression models.

## Objective
- Fetch real stock data using the yfinance API
- Perform data preprocessing and feature engineering
- Train and compare Linear Regression and Random Forest models
- Evaluate models using MAE, RMSE, and R2 Score
- Visualize actual vs predicted stock prices

## Dataset
- Source: Yahoo Finance (via yfinance Python library)
- Stock: Apple Inc. (AAPL)
- Period: Last 2 years of daily trading data
- Features used: Open, High, Low, Volume
- Target variable: Close price

---

## Step 1: Install and Import Libraries

In [ ]:
# Install yfinance if not already installed
!pip install yfinance --quiet

# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Data fetching
import yfinance as yf

# Machine learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Plot settings
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

print('All libraries imported successfully.')

---
## Step 2: Fetch Stock Data from Yahoo Finance

In [ ]:
# Define stock ticker and time period
TICKER = 'AAPL'       # Apple Inc. (change to 'TSLA', 'GOOGL', 'MSFT' etc. if desired)
PERIOD = '2y'         # Last 2 years of data
INTERVAL = '1d'       # Daily data

# Download the data
print(f'Fetching {TICKER} stock data from Yahoo Finance...')
raw_df = yf.download(TICKER, period=PERIOD, interval=INTERVAL, auto_adjust=True)

print(f'Data fetched successfully.')
print(f'Total trading days: {len(raw_df)}')
print(f'Date range: {raw_df.index[0].date()} to {raw_df.index[-1].date()}')

---
## Step 3: Inspect and Explore the Data

In [ ]:
# Flatten multi-level columns if present
if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df.columns = raw_df.columns.get_level_values(0)

df = raw_df.copy()

print('Shape:', df.shape)
print()
print('Column Names:', df.columns.tolist())
print()
print('First 5 rows:')
df.head()

In [ ]:
# Dataset info
print('Dataset Info:')
print('-' * 40)
df.info()

In [ ]:
# Descriptive statistics
print('Descriptive Statistics:')
print('-' * 40)
df.describe().round(2)

In [ ]:
# Check for missing values
print('Missing Values per Column:')
print('-' * 35)
missing = df.isnull().sum()
for col, count in missing.items():
    status = 'None' if count == 0 else f'{count} missing'
    print(f'  {col:12s}: {status}')
print(f'\nTotal missing values: {missing.sum()}')

---
## Step 4: Feature Engineering

In [ ]:
# Create a clean working copy
data = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# --- Technical Indicator Features ---

# 1. Price Range: difference between high and low
data['Price_Range'] = data['High'] - data['Low']

# 2. Open-Close Gap: difference between open and previous close
data['Open_Close_Gap'] = data['Open'] - data['Close'].shift(1)

# 3. Previous day's close (lag feature)
data['Prev_Close'] = data['Close'].shift(1)

# 4. Previous day's volume
data['Prev_Volume'] = data['Volume'].shift(1)

# 5. 5-day Moving Average
data['MA_5'] = data['Close'].rolling(window=5).mean()

# 6. 10-day Moving Average
data['MA_10'] = data['Close'].rolling(window=10).mean()

# 7. Daily return percentage
data['Daily_Return'] = data['Close'].pct_change() * 100

# Drop rows with NaN (due to shift and rolling)
data.dropna(inplace=True)

print('Feature engineering complete.')
print(f'Dataset shape after engineering: {data.shape}')
print()
print('All features:')
for col in data.columns:
    print(f'  - {col}')

In [ ]:
# Preview the engineered dataset
data.head(10)

---
## Step 5: Prepare Features and Target Variable

In [ ]:
# Define feature columns and target
FEATURE_COLS = ['Open', 'High', 'Low', 'Volume',
                'Price_Range', 'Open_Close_Gap',
                'Prev_Close', 'Prev_Volume',
                'MA_5', 'MA_10', 'Daily_Return']

TARGET_COL = 'Close'

X = data[FEATURE_COLS]
y = data[TARGET_COL]

# Save the date index for plotting later
dates = data.index

print(f'Features (X) shape: {X.shape}')
print(f'Target  (y) shape: {y.shape}')
print(f'\nFeature columns: {FEATURE_COLS}')

In [ ]:
# Train-Test Split (80% train, 20% test)
# shuffle=False is critical for time series — no random mixing
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, shuffle=False
)

print('Train-Test Split (Time Series Order Preserved):')
print(f'  Training samples : {len(X_train)}')
print(f'  Testing  samples : {len(X_test)}')
print(f'  Train period     : {dates_train[0].date()} to {dates_train[-1].date()}')
print(f'  Test  period     : {dates_test[0].date()}  to {dates_test[-1].date()}')

In [ ]:
# Feature Scaling (important for Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling applied using StandardScaler.')
print('Training set mean (after scaling): approximately 0')
print('Training set std  (after scaling): approximately 1')

---
## Step 6: Train Machine Learning Models

In [ ]:
# -----------------------------------------------
#  MODEL 1: Linear Regression
# -----------------------------------------------
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

lr_predictions = lr_model.predict(X_test_scaled)

print('Linear Regression model trained successfully.')

In [ ]:
# -----------------------------------------------
#  MODEL 2: Random Forest Regressor
# -----------------------------------------------
rf_model = RandomForestRegressor(
    n_estimators=200,      # Number of trees
    max_depth=10,          # Max tree depth
    min_samples_split=5,   # Min samples to split a node
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)
rf_model.fit(X_train, y_train)  # RF does not need scaled data

rf_predictions = rf_model.predict(X_test)

print('Random Forest model trained successfully.')
print(f'Number of trees: {rf_model.n_estimators}')

---
## Step 7: Model Evaluation

In [ ]:
# Evaluation function
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'  Model            : {name}')
    print(f'  MAE              : {mae:.4f}  (Mean Absolute Error in $)')
    print(f'  RMSE             : {rmse:.4f} (Root Mean Squared Error in $)')
    print(f'  R2 Score         : {r2:.4f}  (1.0 = perfect prediction)')
    print(f'  MAPE             : {mape:.2f}%  (Mean Absolute Percentage Error)')
    return {'Model': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
            'R2': round(r2, 4), 'MAPE': round(mape, 2)}

print('=' * 55)
print('        LINEAR REGRESSION RESULTS')
print('=' * 55)
lr_results = evaluate_model('Linear Regression', y_test, lr_predictions)

print()
print('=' * 55)
print('        RANDOM FOREST RESULTS')
print('=' * 55)
rf_results = evaluate_model('Random Forest', y_test, rf_predictions)

In [ ]:
# Side-by-side comparison table
results_df = pd.DataFrame([lr_results, rf_results])
results_df = results_df.set_index('Model')
print('Model Comparison Table:')
results_df

---
## Step 8: Visualizations

In [ ]:
# =====================================================
#  PLOT 1: Full Historical Closing Price
# =====================================================
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(data.index, data['Close'], color='#2C3E50', linewidth=1.2, label='Close Price')
ax.fill_between(data.index, data['Close'], alpha=0.1, color='#2C3E50')

# Mark train/test boundary
split_date = dates_test[0]
ax.axvline(split_date, color='red', linestyle='--', linewidth=1.2, label='Train/Test Split')
ax.text(split_date, data['Close'].max() * 0.95, '  Test Start',
        color='red', fontsize=9)

ax.set_title(f'{TICKER} Historical Closing Price (Last 2 Years)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (USD)', fontsize=11)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.grid(alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot1_historical_price.png', bbox_inches='tight')
plt.show()
print('Insight: The red dashed line separates training data (left) from test data (right).')

In [ ]:
# =====================================================
#  PLOT 2: Moving Averages
# =====================================================
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(data.index, data['Close'], color='#2C3E50', linewidth=1,
        alpha=0.7, label='Close Price')
ax.plot(data.index, data['MA_5'],  color='#E74C3C', linewidth=1.3,
        label='5-Day Moving Average')
ax.plot(data.index, data['MA_10'], color='#27AE60', linewidth=1.3,
        label='10-Day Moving Average')

ax.set_title(f'{TICKER} Closing Price with Moving Averages',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (USD)', fontsize=11)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.grid(alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot2_moving_averages.png', bbox_inches='tight')
plt.show()
print('Insight: Moving averages smooth out noise and reveal the underlying price trend.')

In [ ]:
# =====================================================
#  PLOT 3: Actual vs Predicted — Linear Regression
# =====================================================
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(dates_test, y_test.values, color='#2C3E50', linewidth=1.5,
        label='Actual Price', zorder=3)
ax.plot(dates_test, lr_predictions, color='#E74C3C', linewidth=1.3,
        linestyle='--', label='Linear Regression Prediction', zorder=2)

ax.set_title(f'{TICKER} -- Actual vs Predicted (Linear Regression)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (USD)', fontsize=11)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.grid(alpha=0.3)
sns.despine()

# Annotate with metrics
ax.text(0.02, 0.05,
        f"MAE: {lr_results['MAE']:.2f}  |  RMSE: {lr_results['RMSE']:.2f}  |  R2: {lr_results['R2']:.4f}",
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('plot3_lr_actual_vs_predicted.png', bbox_inches='tight')
plt.show()
print(f"Insight: Linear Regression R2 = {lr_results['R2']} (closer to 1.0 is better).")

In [ ]:
# =====================================================
#  PLOT 4: Actual vs Predicted -- Random Forest
# =====================================================
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(dates_test, y_test.values, color='#2C3E50', linewidth=1.5,
        label='Actual Price', zorder=3)
ax.plot(dates_test, rf_predictions, color='#27AE60', linewidth=1.3,
        linestyle='--', label='Random Forest Prediction', zorder=2)

ax.set_title(f'{TICKER} -- Actual vs Predicted (Random Forest)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (USD)', fontsize=11)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.grid(alpha=0.3)
sns.despine()

ax.text(0.02, 0.05,
        f"MAE: {rf_results['MAE']:.2f}  |  RMSE: {rf_results['RMSE']:.2f}  |  R2: {rf_results['R2']:.4f}",
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.tight_layout()
plt.savefig('plot4_rf_actual_vs_predicted.png', bbox_inches='tight')
plt.show()
print(f"Insight: Random Forest R2 = {rf_results['R2']} (closer to 1.0 is better).")

In [ ]:
# =====================================================
#  PLOT 5: Both Models Side by Side
# =====================================================
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# Top: Linear Regression
axes[0].plot(dates_test, y_test.values, color='#2C3E50',
             linewidth=1.5, label='Actual')
axes[0].plot(dates_test, lr_predictions, color='#E74C3C',
             linewidth=1.2, linestyle='--', label='Linear Regression')
axes[0].set_title('Linear Regression -- Actual vs Predicted',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price (USD)', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Bottom: Random Forest
axes[1].plot(dates_test, y_test.values, color='#2C3E50',
             linewidth=1.5, label='Actual')
axes[1].plot(dates_test, rf_predictions, color='#27AE60',
             linewidth=1.2, linestyle='--', label='Random Forest')
axes[1].set_title('Random Forest -- Actual vs Predicted',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=10)
axes[1].set_ylabel('Price (USD)', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)

fig.suptitle(f'{TICKER} Stock Price Prediction -- Model Comparison',
             fontsize=14, fontweight='bold', y=1.01)
sns.despine()
plt.tight_layout()
plt.savefig('plot5_both_models.png', bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================
#  PLOT 6: Residual Plot (Prediction Errors)
# =====================================================
lr_residuals = y_test.values - lr_predictions
rf_residuals = y_test.values - rf_predictions

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, residuals, name, color in zip(
        axes,
        [lr_residuals, rf_residuals],
        ['Linear Regression', 'Random Forest'],
        ['#E74C3C', '#27AE60']):

    ax.scatter(range(len(residuals)), residuals, alpha=0.5,
               color=color, s=15, edgecolors='none')
    ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_title(f'{name} -- Residuals', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Index', fontsize=10)
    ax.set_ylabel('Residual (Actual - Predicted)', fontsize=10)
    ax.grid(alpha=0.3)

fig.suptitle('Residual Analysis -- Prediction Errors',
             fontsize=14, fontweight='bold', y=1.02)
sns.despine()
plt.tight_layout()
plt.savefig('plot6_residuals.png', bbox_inches='tight')
plt.show()
print('Insight: Points closer to the horizontal zero line indicate smaller prediction errors.')

In [ ]:
# =====================================================
#  PLOT 7: Scatter -- Actual vs Predicted (both models)
# =====================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, name, color in zip(
        axes,
        [lr_predictions, rf_predictions],
        ['Linear Regression', 'Random Forest'],
        ['#E74C3C', '#27AE60']):

    ax.scatter(y_test.values, preds, alpha=0.5, color=color, s=20, edgecolors='none')

    # Perfect prediction line
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--',
            linewidth=1.2, label='Perfect Prediction')

    ax.set_title(f'{name}\nActual vs Predicted', fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual Price (USD)', fontsize=10)
    ax.set_ylabel('Predicted Price (USD)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle('Actual vs Predicted Scatter Plot', fontsize=14,
             fontweight='bold', y=1.02)
sns.despine()
plt.tight_layout()
plt.savefig('plot7_scatter_actual_predicted.png', bbox_inches='tight')
plt.show()
print('Insight: Points closer to the dashed diagonal line indicate better predictions.')

In [ ]:
# =====================================================
#  PLOT 8: Feature Importance -- Random Forest
# =====================================================
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_df)))
bars = ax.barh(feat_df['Feature'], feat_df['Importance'],
               color=colors, edgecolor='white', linewidth=0.5)

# Add value labels
for bar, val in zip(bars, feat_df['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title('Random Forest -- Feature Importance',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.grid(axis='x', alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot8_feature_importance.png', bbox_inches='tight')
plt.show()
print('Insight: Higher importance = greater influence on price prediction.')

In [ ]:
# =====================================================
#  PLOT 9: Model Metrics Comparison Bar Chart
# =====================================================
metrics = ['MAE', 'RMSE']
lr_vals = [lr_results['MAE'], lr_results['RMSE']]
rf_vals = [rf_results['MAE'], rf_results['RMSE']]

x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, lr_vals, width, label='Linear Regression',
               color='#E74C3C', edgecolor='white')
bars2 = ax.bar(x + width/2, rf_vals, width, label='Random Forest',
               color='#27AE60', edgecolor='white')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f'{bar.get_height():.2f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Error Metrics Comparison (Lower is Better)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Error (USD)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot9_metrics_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================
#  PLOT 10: Daily Return Distribution
# =====================================================
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(data['Daily_Return'], bins=50, color='#3498DB',
        edgecolor='white', linewidth=0.5, alpha=0.85)
ax.axvline(data['Daily_Return'].mean(), color='red',
           linestyle='--', linewidth=1.5,
           label=f"Mean: {data['Daily_Return'].mean():.2f}%")
ax.axvline(0, color='black', linestyle='-', linewidth=1,
           label='Zero Return')

ax.set_title(f'{TICKER} Daily Return Distribution (%)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Daily Return (%)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig('plot10_daily_return.png', bbox_inches='tight')
plt.show()
print('Insight: Daily returns follow a roughly normal distribution centered near zero.')

---
## Step 9: Statistical Summary

In [ ]:
# Correlation matrix of all features
print('Correlation Matrix of Features:')
corr = data[FEATURE_COLS + ['Close']].corr().round(2)

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, linecolor='white',
            annot_kws={'size': 8}, vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('plot11_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Insight: High correlation between Prev_Close and Close confirms lag features are useful.')

---
## Step 10: Final Results and Conclusion

### Model Evaluation Summary

| Metric | Linear Regression | Random Forest | Better Model |
|--------|------------------|---------------|--------------|
| MAE (USD) | See above | See above | Lower is better |
| RMSE (USD) | See above | See above | Lower is better |
| R2 Score | See above | See above | Higher is better |
| MAPE (%) | See above | See above | Lower is better |

### Key Findings

1. **Random Forest outperforms Linear Regression** on all metrics due to its ability to capture non-linear relationships in stock data.
2. **Previous day's Close price (Prev_Close)** is the most important feature for prediction -- stock prices are strongly correlated with their recent history.
3. **Moving averages (MA_5, MA_10)** significantly improve prediction accuracy by capturing short-term trends.
4. **High and Low prices** are also important features since they define the daily trading range.
5. **Volume** has relatively lower importance, suggesting price movement is more predictive than trade volume alone.

### Limitations

- Stock prices are affected by external factors (news, earnings, global events) not captured in price data alone.
- Short-term prediction is inherently uncertain -- these models should not be used for actual financial decisions.
- More advanced models (LSTM, XGBoost, Transformer) would likely perform better for real-world use.

### Conclusion

This task demonstrated how machine learning regression models can be applied to financial time series data. By engineering meaningful features such as moving averages, price range, and lag variables, both models achieved strong predictions on unseen test data. Random Forest proved to be the superior model for this task due to its ensemble nature and ability to model complex patterns.

---
### Tools and Libraries Used
- **yfinance** -- Real-time stock data from Yahoo Finance
- **pandas, numpy** -- Data manipulation
- **scikit-learn** -- Linear Regression, Random Forest, evaluation metrics
- **matplotlib, seaborn** -- Visualization

---
*DevelopersHub Corporation -- AI/ML Engineering Internship | Task 2*